In [2]:
# setup
import numpy as np #numbers python library
from scipy.signal import butter, lfilter #general signal processing python library
import soundfile as sf #library to read and write sound files
import librosa #audio and music processing for python
import os
import random

In [8]:
import shutil

folder = "dataset"
shutil.rmtree(folder)       # deletes folder and everything inside
os.makedirs(folder)          # recreate the empty folder

#create dataset folders
folders_nmf = [
    'dataset/static',
    'dataset/dynamic',
    'dataset/static_kick',
    'dataset/static_snare',
    'dataset/dynamic_kick',
    'dataset/dynamic_snare'
]

for folder in folders_nmf:
    os.makedirs(folder, exist_ok=True)

In [3]:
def generate_time_vector(duration, sample_rate):
    """
    Returns a time vector from 0 to duration (seconds).
    """
    num_samples = int(duration * sample_rate)
    return np.linspace(0, duration, num_samples, endpoint=False)

def exponential_decay_envelope(t, tau):
    """
    A(t) = exp(-t / tau)
    """
    return np.exp(-t / tau)

def smooth_envelope(n_samples, fade_in_samples, fade_out_samples):
    """
    Create a smooth trapezoidal envelope with raised-cosine fades.
    Eliminates spectral splatter from abrupt onset/offset.
    """
    env = np.ones(n_samples)
    # Raised-cosine fade in
    if fade_in_samples > 0:
        env[:fade_in_samples] = 0.5 * (1 - np.cos(np.pi * np.arange(fade_in_samples) / fade_in_samples))
    # Raised-cosine fade out
    if fade_out_samples > 0:
        env[-fade_out_samples:] = 0.5 * (1 + np.cos(np.pi * np.arange(fade_out_samples) / fade_out_samples))
    return env

def bandpass_filter(signal, low_freq, high_freq, sample_rate, order=4):
    nyquist = sample_rate / 2
    low = low_freq / nyquist
    high = high_freq / nyquist

    b, a = butter(order, [low, high], btype='band')
    return lfilter(b, a, signal)

def highpass_filter(signal, cutoff_freq, sample_rate, order=4):
    nyquist = sample_rate / 2
    cutoff = cutoff_freq / nyquist

    b, a = butter(order, cutoff, btype='high')
    return lfilter(b, a, signal)

def pitch_envelope(t, f0, tau):
    """
    Generate an exponential pitch envelope.
    
    Args:
        t: time vector
        f0: initial frequency (Hz)
        tau: time constant for exponential decay
    
    Returns:
        freq_env: array of frequencies over time
    """
    return f0 * np.exp(-t / tau)

def time_varying_bandpass(noise, freq_env, sample_rate, frame_size=512, bandwidth_factor=0.3):
    """
    Apply a time-varying bandpass filter where center frequency changes over time.
    
    Args:
        noise: input signal
        freq_env: array of center frequencies over time
        sample_rate: sample rate
        frame_size: size of processing frames
        bandwidth_factor: relative bandwidth (0.3 = ±30% around center)
    
    Returns:
        filtered signal
    """
    output = np.zeros_like(noise)
    nyq = sample_rate / 2.0
    
    for start in range(0, len(noise), frame_size):
        end = min(start + frame_size, len(noise))
        center = freq_env[start]
        
        # make a narrow band around the center
        low_hz = max(50.0, center * (1 - bandwidth_factor))
        high_hz = min(nyq - 1.0, center * (1 + bandwidth_factor))
        
        if low_hz >= high_hz:
            seg_filtered = np.zeros(end - start)
        else:
            low = low_hz / nyq
            high = high_hz / nyq
            b, a = butter(2, [low, high], btype='band')
            seg_filtered = lfilter(b, a, noise[start:end])
        
        output[start:end] = seg_filtered
    
    return output

def synthesize_kick_static(duration, sample_rate, freq=60, tau=0.05):
    """
    Static kick: pure sine at fixed frequency with smooth envelope.
    No spectral evolution — ideal for NMF's single-column template.
    """
    t = generate_time_vector(duration, sample_rate)
    n = len(t)
    # Smooth fade-in (10ms) and fade-out (50ms) to prevent spectral splatter
    fade_in = int(0.01 * sample_rate)
    fade_out = int(0.05 * sample_rate)
    env = smooth_envelope(n, fade_in, fade_out)
    signal = env * np.sin(2 * np.pi * freq * t)
    return signal

def synthesize_kick_dynamic(duration, sample_rate, f0=150, tau_f=0.05, tau_amp=0.05):
    t = generate_time_vector(duration, sample_rate)
    envelope = exponential_decay_envelope(t, tau_amp)

    # Generate pitch envelope (exponential pitch drop)
    freq = pitch_envelope(t, f0, tau_f)

    # Phase = integral of frequency
    phase = 2 * np.pi * np.cumsum(freq) / sample_rate
    signal = envelope * np.sin(phase)
    return signal

def synthesize_snare_static(duration, sample_rate, low=5000, high=9000, tau=0.1):
    """
    Static snare: bandpass noise at fixed high-frequency band with smooth envelope.
    No spectral evolution — ideal for NMF's single-column template.
    Wide gap from kick (60 Hz vs 5-9 kHz) ensures clean spectral separation.
    """
    t = generate_time_vector(duration, sample_rate)
    n = len(t)
    # Smooth fade-in (10ms) and fade-out (50ms) to prevent spectral splatter
    fade_in = int(0.01 * sample_rate)
    fade_out = int(0.05 * sample_rate)
    env = smooth_envelope(n, fade_in, fade_out)

    noise = np.random.randn(n)
    noise = bandpass_filter(noise, low, high, sample_rate)
    return env * noise

def synthesize_snare_dynamic(duration, sample_rate):
    t = generate_time_vector(duration, sample_rate)

    # Transient envelope (fast)
    env_transient = exponential_decay_envelope(t, 0.01)
    # Tail envelope (slow)
    env_tail = exponential_decay_envelope(t, 0.15)

    noise = np.random.randn(len(t))

    transient = highpass_filter(noise, 4000, sample_rate)

    # Generate pitch envelope
    freq_env = pitch_envelope(t, f0=3000.0, tau=0.08)
    
    # Apply time-varying bandpass
    tail = time_varying_bandpass(noise, freq_env, sample_rate, frame_size=512, bandwidth_factor=0.3)

    return env_transient * transient + env_tail * tail

def generate_event_times(rate, mixture_length):
    """
    Poisson process event times (seconds).
    """
    times = []
    t = 0
    while t < mixture_length:
        t += np.random.exponential(1 / rate)
        if t < mixture_length:
            times.append(t)
    return times

def add_event_to_track(track, event, onset_time, sample_rate):
    onset_sample = int(onset_time * sample_rate)
    end_sample = onset_sample + len(event)

    if end_sample > len(track):
        end_sample = len(track)
        event = event[:end_sample - onset_sample]

    track[onset_sample:end_sample] += event

def normalize_audio(signal):
    max_val = np.max(np.abs(signal))
    if max_val > 0:
        return signal / max_val
    return signal

def generate_mixture(condition, sample_rate=22050, length=15.0):
    total_samples = int(sample_rate * length)
    kick_track = np.zeros(total_samples)
    snare_track = np.zeros(total_samples)

    # Event rates
    kick_times = generate_event_times(rate=2, mixture_length=length)
    snare_times = generate_event_times(rate=2, mixture_length=length)

    for t in kick_times:
        if condition == "static":
            event = synthesize_kick_static(random.choice([0.05, 0.5, 2]), sample_rate)
        else:
            event = synthesize_kick_dynamic(0.2, sample_rate)
        add_event_to_track(kick_track, event, t, sample_rate)

    for t in snare_times:
        if condition == "static":
            event = synthesize_snare_static(random.choice([0.05, 0.5, 2]), sample_rate)
        else:
            event = synthesize_snare_dynamic(0.2, sample_rate)
        add_event_to_track(snare_track, event, t, sample_rate)

    mixture = kick_track + snare_track
    mixture = normalize_audio(mixture)
    kick_track = normalize_audio(kick_track)
    snare_track = normalize_audio(snare_track)

    return mixture, kick_track, snare_track

def save_mixture(mixture, index, condition, sample_rate):
    prefix = f"dataset/{condition}/{condition}_mix_{index:03d}"
    sf.write(prefix + ".wav", mixture, sample_rate)

def save_snares(snare, index, condition, sample_rate):
    prefix = f"dataset/{condition}_snare/{condition}_snare_{index:03d}"
    sf.write(prefix + ".wav", snare, sample_rate)

def save_kick(kick, index, condition, sample_rate):
    prefix = f"dataset/{condition}_kick/{condition}_kick_{index:03d}"
    sf.write(prefix + ".wav", kick, sample_rate)

def generate_dataset(num_mixtures, sample_rate):
    for condition in ["static", "dynamic"]:
        for i in range(num_mixtures):
            mix, kick, snare = generate_mixture(condition, sample_rate)
            save_mixture(mix, i, condition, sample_rate)
            save_snares(snare, i, condition, sample_rate)
            save_kick(kick, i, condition, sample_rate)

In [4]:
generate_dataset(20, 22050)